# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Target:** `is_declining` (`trend_direction == "down"`) — the observed label this whole lane is
built around since Week 2.

**Method:** Logistic Regression first (readable, a real coefficient story), then Random Forest
(stronger, handles nonlinear signal combos) — matching the skill's "yes/no with an observed
label → readable, then stronger" rule. Permutation importance afterward for interpretation.

**A leak found before any modeling, not assumed:** `impressions_last_30d` vs.
`impressions_prev_30d` turns out to reconstruct `trend_pct` almost exactly (correlation
0.99999...) — proven below, not guessed. So both columns, and the same-family
`clicks_last_30d/prev_30d` and `sessions_last_30d/prev_30d`, are excluded entirely. The
remaining 90-day aggregate features (`impressions_90d`, `ctr`, `avg_position`, etc.) still
partially overlap the label's own 30-day window, so this model is framed honestly as a
**current-state diagnostic** ("does this page's present profile look like other declining
pages?") — decision-support, not a clean forward forecast.


In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# prove the leak before excluding it -- don't just assert it
mask = df["impressions_prev_30d"] > 0
implied_pct_change = (df.loc[mask, "impressions_last_30d"] - df.loc[mask, "impressions_prev_30d"]) / df.loc[mask, "impressions_prev_30d"] * 100
leak_corr = implied_pct_change.corr(df.loc[mask, "trend_pct"])
print(f"correlation between implied impressions % change and trend_pct: {leak_corr:.6f}")
print("-> trend_pct IS this ratio. impressions_last_30d / impressions_prev_30d excluded, and the")
print("   same-family clicks/sessions last_30d & prev_30d columns excluded for consistency.")


correlation between implied impressions % change and trend_pct: 1.000000
-> trend_pct IS this ratio. impressions_last_30d / impressions_prev_30d excluded, and the
   same-family clicks/sessions last_30d & prev_30d columns excluded for consistency.


In [2]:
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count", "content_age_days",
    "age_tier_order", "days_since_last_update", "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
categorical_features = ["competition_level", "content_type", "main_intent"]

# missingness follows content_type (per the flyrank-data skill) -- flag before imputing, don't blind-fillna
for c in ["word_count", "char_count"]:
    df[f"has_{c}"] = df[c].notna().astype(int)
numeric_features += ["has_word_count", "has_char_count"]

excluded = ["trend_pct", "trend_direction", "is_declining", "content_id", "client_id",
            "impressions_last_30d", "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d",
            "sessions_last_30d", "sessions_prev_30d"]

print("feature count:", len(numeric_features) + len(categorical_features))
print("excluded (label + leakage family):", excluded)


feature count: 28
excluded (label + leakage family): ['trend_pct', 'trend_direction', 'is_declining', 'content_id', 'client_id', 'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id`.** A row-random split would let the same client appear in both train
and test, so the model could partly memorize client-specific baseline behavior rather than learn
generalizable content signals — exactly the split the `flyrank-data` skill recommends
(`client_id` for grouping, never as a feature). 70/30 split, one held-out fold.


In [3]:
from sklearn.model_selection import GroupShuffleSplit

X = df[numeric_features + categorical_features].copy()
y = df["is_declining"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
ytr, yte = y.iloc[train_idx], y.iloc[test_idx]

print("train rows:", len(Xtr), "| test rows:", len(Xte))
print("client overlap between train and test:", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))
print("train base rate:", round(ytr.mean(), 3), "| test base rate:", round(yte.mean(), 3))


train rows: 19166 | test rows: 10834
client overlap between train and test: 0
train base rate: 0.532 | test base rate: 0.559


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Baseline = the exact Week-4 rule (CTR below half its position tier's typical rate, on visible
pages), scored on this same test fold — no retraining needed, it's a rule, not a fit. All three
methods evaluated at precision@K / recall@K (K = top 10% of the test fold, same "editor's
sprint" framing as Week 2) plus ROC-AUC, on the same split, same metric.


In [4]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

pre_lr = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])
logreg = Pipeline([("pre", pre_lr), ("clf", LogisticRegression(max_iter=2000, random_state=42))])
logreg.fit(Xtr, ytr)
logreg_proba = logreg.predict_proba(Xte)[:, 1]

pre_rf = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])
rf = Pipeline([("pre", pre_rf), ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1))])
rf.fit(Xtr, ytr)
rf_proba = rf.predict_proba(Xte)[:, 1]

print("models fit. random_state=42 throughout for reproducibility.")


models fit. random_state=42 throughout for reproducibility.


In [5]:
# baseline: the Week-4 rule, computed on the full df (no fitting -- a rule needs no training),
# then subset to this test fold only, for a same-split comparison
expected_ctr_for_tier = df.groupby("position_tier")["ctr"].transform("mean")
underperforms_ctr = (df["ctr"] < expected_ctr_for_tier * 0.5).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = underperforms_ctr * visible * df["impressions_90d"]
baseline_score_test = df.loc[Xte.index, "baseline_score"].values

def precision_recall_at_k(scores, y_true, k):
    order = np.argsort(-scores)
    top_k = order[:k]
    y_true_arr = np.asarray(y_true)
    hits = y_true_arr[top_k].sum()
    return hits / k, hits / y_true_arr.sum()

K = int(0.10 * len(Xte))  # top 10% of the test fold, same editor-sprint framing as Week 2
print("K (top 10% of test fold):", K)

rows = []
for name, scores in [("baseline_rule (Week 4)", baseline_score_test),
                      ("logistic_regression", logreg_proba),
                      ("random_forest", rf_proba)]:
    p, r = precision_recall_at_k(scores, yte, K)
    auc = roc_auc_score(yte, scores)
    rows.append({"method": name, "precision_at_K": round(p, 3), "recall_at_K": round(r, 3), "roc_auc": round(auc, 3)})

comparison = pd.DataFrame(rows)
print(f"test-fold base rate (naive 'predict all declining'): {yte.mean():.3f}")
comparison


K (top 10% of test fold): 1083
test-fold base rate (naive 'predict all declining'): 0.559


,method,precision_at_K,recall_at_K,roc_auc
0,baseline_rule (Week 4),0.596,0.107,0.554
1,logistic_regression,0.681,0.122,0.601
2,random_forest,0.657,0.117,0.617


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Random Forest wins on ROC-AUC, Logistic Regression edges it on precision@K — reporting both,
per the skill's rule ("if the model wins on one metric and loses on another, report both, that
IS the finding"). Permutation importance on the Random Forest below, then concrete wrong cases.


In [6]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, Xte, yte, n_repeats=8, random_state=42, n_jobs=-1, scoring="roc_auc")
feat_names = numeric_features + categorical_features
importance_df = pd.DataFrame({
    "feature": feat_names,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

importance_df.head(10)


,feature,importance_mean,importance_std
16,days_with_impressions,0.036262,0.002755
5,content_age_days,0.010204,0.002204
19,avg_position,0.006617,0.000631
8,impressions_90d,0.006604,0.001457
18,ctr,0.004438,0.000483
9,clicks_90d,0.004349,0.000477
21,scroll_rate,0.003927,0.001884
17,days_with_sessions,0.003373,0.000798
13,engaged_sessions_90d,0.001686,0.000417
10,pageviews_90d,0.001544,0.000481


**Top feature: `days_with_impressions`** — how many days in the window the page had any
visibility at all. This makes sense: sustained presence is the opposite of decline, not a
suspiciously perfect proxy for the label itself. `content_age_days` is #2, consistent with
Week 4's finding that *newer*, not older, content declines more here. No single feature dominates
(max importance 0.036 on the ROC-AUC scale) — a good sign against leftover leakage.


In [7]:
test_df = df.loc[Xte.index].copy()
test_df["proba"] = rf_proba
test_df["pred"] = (rf_proba >= 0.5).astype(int)
test_df["actual"] = yte.values

review_cols = ["content_id", "content_type", "avg_position", "ctr", "impressions_90d", "content_age_days", "proba"]

false_neg = test_df[(test_df["actual"] == 1) & (test_df["pred"] == 0)].sort_values("proba").head(2)
false_pos = test_df[(test_df["actual"] == 0) & (test_df["pred"] == 1)].sort_values("proba", ascending=False).head(2)

print("FALSE NEGATIVES (actually declining, model said no):")
print(false_neg[review_cols].to_string(index=False))
print()
print("FALSE POSITIVES (not declining, model said yes):")
print(false_pos[review_cols].to_string(index=False))


FALSE NEGATIVES (actually declining, model said no):
          content_id    content_type  avg_position  ctr  impressions_90d  content_age_days    proba
content_3a4e24a3a6a8 keyword article           4.0  0.0                1               332 0.073742
content_7bc32bc1df59 keyword article           0.0  0.0                1               238 0.126436

FALSE POSITIVES (not declining, model said yes):
          content_id    content_type  avg_position  ctr  impressions_90d  content_age_days    proba
content_884c401ce126 keyword article          23.1  0.0              101               174 0.839509
content_0b47dae0c7f9 keyword article          23.1  0.0             1191               238 0.834739


**Why these are hard:**

1. **The two false negatives** both have `impressions_90d = 1` and `ctr = 0` — essentially no
   traffic at all in either direction. There's almost no signal to separate "just launched, hasn't
   started earning clicks yet" from "declined all the way to near-zero." One even shows
   `avg_position = 0`, which (per the data skill) means *no position data*, not rank zero — the
   model is working from a genuinely empty signal here, not failing to find one.
2. **The two false positives** both sit at `avg_position = 23.1` with `ctr = 0` but real
   visibility (101 and 1,191 impressions). These look like moderately-ranked pages that simply
   haven't earned a click yet, not pages actively losing ground — the model may be over-weighting
   "zero clicks" as a decline signal when it's sometimes just a slow starter.

Both error types cluster around near-zero-click content — the model's real blind spot is *low-
signal* pages, not any single feature or tier.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.